# Setup

In [ ]:
import os, gc, json, time, warnings
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import pearsonr, spearmanr, ttest_rel
from numpy.fft import fft, ifft
from sklearn.utils import check_random_state
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS = "../4_results"

# ---- atlas / threshold: results go to ../4_results/2_classification_{ATLAS}_{ALPHA} ----
ATLAS = "Shen"            # "Shen" or "Schaefer": must match ATLAS_NAME used in a_Arousal_FC to make the FC files
ALPHA = 0.05              # feature-selection p threshold. Changing this changes the output directory, so don't worry about overwriting!

# ---- what to run -------------------------------------------------------------
RESHAPE          = 1      # Rebuild Cut Arrays (necessary on first time)
RUN_PREDICTIONS  = True
RUN_PERMUTATIONS = True
RUN_PAIRED_TESTS = True
RUN_FIGURES      = True
RUN_CPM_EXPORT   = True

# ---- settings ----------------------------------------------------------------
N_PERM          = 10000
SEED            = None
FAST_NULL       = True
SHOW_PERM_PLOTS = True
FAST_CORR       = True
LOAD_MMAP       = True
RECOMPUTE_CACHE = False

CLS_DIR   = os.path.join(RESULTS, f"2_classification_{ATLAS}_{ALPHA:g}")
FIG_DIR   = os.path.join(CLS_DIR, "figures")
CPM_DIR   = os.path.join(RESULTS, "CPM")
N_ROIS_EXPECTED = {"Shen": 268, "Schaefer": 208}[ATLAS]
SFX       = "" if ATLAS == "Shen" else f"_{ATLAS}"   # FC / cut file suffix: Shen none, Schaefer "_Schaefer" (as in a)
CACHE_DIR = os.path.join(RESULTS, "1_arousal", "_rmatrix_cache", *([] if ATLAS == "Shen" else [ATLAS]))   # per-subject correlations (atlas-specific, alpha-free)
PERM_BASE = f"model5_p{f'{ALPHA:g}'.replace('0.', '', 1)}_mean"       # model5_p05_mean for ALPHA = 0.05
SLEEPWAKE_DIR = CLS_DIR   # sleep/wake results (16–19) are made in this notebook, before the figures
print(f"Writing classification results to: {CLS_DIR}")


def res(*parts):
    """Path under ../4_results."""
    return os.path.join(RESULTS, *parts)


def save_npy(path, arr):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.save(path, arr)


def save_csv(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)


def save_txt(path, text):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(text)

# Manually remove Motion Outliers and Trim dFC Nans on Time Course Edges

In [ ]:
DATASETS = {
    # name: raw inputs, cleaning, and the *_Cut files the analysis reads. Arousal comes from a and is the same for
    # every atlas; FC and cut files carry SFX (Shen: none, Schaefer: "_Schaefer"), so the two atlases never overwrite
    # each other
    "EEG_Rest": dict(
        raw_arousal="1_arousal/1_EEG_Rest/Convolved_EEG_Rest_Arousal.npy",
        raw_fc=f"1_arousal/1_EEG_Rest/EEG_FC_array{SFX}.npy",
        trim=7, tile=None, remove=[1, 7],
        fc=f"1_arousal/1_EEG_Rest/EEG_Rest_FC_Cut{SFX}.npy",
        arousal=f"1_arousal/1_EEG_Rest/EEG_Rest_Arousal_Cut{SFX}.npy"),
    "Pupil_Rest": dict(
        raw_arousal="1_arousal/2_Pupil_Rest/Convolved_Pupil_Rest_Arousal.npy",
        raw_fc=f"1_arousal/2_Pupil_Rest/Pupil_FC_array{SFX}.npy",
        trim=15, tile=None, remove=[2, 18],
        fc=f"1_arousal/2_Pupil_Rest/Pupil_Rest_FC_Cut{SFX}.npy",
        arousal=f"1_arousal/2_Pupil_Rest/Pupil_Rest_Arousal_Cut{SFX}.npy"),
    "Pupil_Story": dict(
        raw_arousal="1_arousal/3_Pupil_Story/Convolved_Pupil_Story_Arousal.npy",
        raw_fc=f"1_arousal/3_Pupil_Story/Pupil_Story_FC_array{SFX}.npy",
        trim=None, tile=22, remove=[0, 20],
        fc=f"1_arousal/3_Pupil_Story/Pupil_Story_FC_Cut{SFX}.npy",
        arousal=f"1_arousal/3_Pupil_Story/Pupil_Story_Arousal_Cut{SFX}.npy"),
    "FilmFest": dict(
        raw_arousal="1_arousal/4_FilmFestival/Filmfest_Event_Arousal.npy",
        raw_fc=f"1_arousal/4_FilmFestival/FilmFestival_FC_array{SFX}.npy",
        trim=None, tile=20, remove=[2, 3, 4, 5, 14],
        fc=f"1_arousal/4_FilmFestival/FilmFestival_FC_Cut{SFX}.npy",
        arousal=f"1_arousal/4_FilmFestival/FilmFestival_Arousal_Cut{SFX}.npy"),
    "Sherlock": dict(
        raw_arousal="1_arousal/5_Sherlock/Sherlock_Event_Arousal.npy",
        raw_fc=f"1_arousal/5_Sherlock/Sherlock_FC_array{SFX}.npy",
        trim=None, tile=15, remove=[12],
        fc=f"1_arousal/5_Sherlock/Sherlock_FC_Cut{SFX}.npy",
        arousal=f"1_arousal/5_Sherlock/Sherlock_Arousal_Cut{SFX}.npy"),
}


def build_cut(name):
    """Same steps as the old `if reshape:` blocks (trim → drop subjects → reshape → save).
    Skipped when the cut files are already newer than the ab-notebook files they come from."""
    cfg = DATASETS[name]
    raw = [res(cfg["raw_arousal"]), res(cfg["raw_fc"])]
    cut = [res(cfg["fc"]), res(cfg["arousal"])]
    if all(os.path.exists(c) for c in cut) and \
            min(os.path.getmtime(c) for c in cut) > max(os.path.getmtime(r) for r in raw):
        print(f"{name}: cut files are up to date")
        return
    arousal = np.load(res(cfg["raw_arousal"]))
    if cfg["tile"] is not None:                        # event/story datasets: same arousal for every subject
        arousal = np.tile(arousal, (cfg["tile"], 1))
    fc = np.load(res(cfg["raw_fc"]), mmap_mode="r")
    n_subj = arousal.shape[0] if cfg["tile"] is None else cfg["tile"]
    keep = np.ones(n_subj, dtype=bool)
    keep[cfg["remove"]] = False

    arousal_cut = arousal[keep].copy()
    if cfg["trim"] is not None:                        # rest datasets: trim TRs per run, concatenate runs
        t = cfg["trim"]
        _, n_runs, n_tr, n_feat = fc.shape
        kept = np.array(fc[:, :, t:-t, :][keep], copy=True)
        fc_cut = kept.reshape(kept.shape[0], n_runs * (n_tr - 2 * t), n_feat)
        del kept
    else:
        fc_cut = np.array(fc[keep], copy=True)
    print(f"{name}: arousal cut {arousal_cut.shape}, FC cut {fc_cut.shape}")
    save_npy(res(cfg["fc"]), fc_cut)
    save_npy(res(cfg["arousal"]), arousal_cut)
    del fc, fc_cut, arousal, arousal_cut
    gc.collect()


# warn if the FC files were made with a different atlas than ATLAS (the ab notebook records this)
_log = res("1_arousal", "FC_atlas.json")
if os.path.exists(_log):
    with open(_log) as _f:
        _fc_atlas = json.load(_f)
    _bad = {k: v for k, v in _fc_atlas.items() if k in {f"{d}{SFX}" for d in DATASETS} and v != ATLAS}
    if _bad:
        print(f"⚠️ ATLAS = '{ATLAS}' but these FC files were made with another atlas: {_bad}")

if RESHAPE:
    for _name in DATASETS:
        build_cut(_name)

AROUSAL, FC, FC_FILES = {}, {}, {}
for _name, _cfg in DATASETS.items():
    FC_FILES[_name] = (res(_cfg["fc"]), res(_cfg["arousal"]))
    AROUSAL[_name] = np.load(FC_FILES[_name][1])
    FC[_name] = np.load(FC_FILES[_name][0], mmap_mode="r" if LOAD_MMAP else None)
    print(f"{_name:12s} arousal {AROUSAL[_name].shape}  FC {FC[_name].shape} {FC[_name].dtype}")

N_FEATURES = FC["EEG_Rest"].shape[2]
N_ROIS = int((1 + np.sqrt(1 + 8 * N_FEATURES)) / 2)
if N_ROIS != N_ROIS_EXPECTED:
    raise ValueError(f"The FC files have {N_ROIS} ROIs, expected {N_ROIS_EXPECTED} for ATLAS = '{ATLAS}'. "
                     f"Run a_Arousal_FC with ATLAS_NAME = '{ATLAS}' first.")
ROI_PAIRS = list(combinations(range(1, N_ROIS + 1), 2))

## Per-edge mean FC and FC variability → ../4_results/CPM/

In [ ]:
# Per-edge FC summaries for the reference-space analysis
# mean = mean FC over time windows (events for FilmFest/Sherlock), sd = FC variability over those windows
os.makedirs(CPM_DIR, exist_ok=True)
for name in DATASETS:
    fc = FC[name]                                  # (subjects, time, edges), memory-mapped
    mean_fc = np.empty((fc.shape[0], fc.shape[2]), dtype=np.float32)
    sd_fc = np.empty_like(mean_fc)
    with warnings.catch_warnings():                # an all-NaN edge just gives NaN
        warnings.simplefilter("ignore", RuntimeWarning)
        for s in range(fc.shape[0]):               # one subject at a time to keep memory low
            x = np.asarray(fc[s], dtype=np.float64)
            mean_fc[s] = np.nanmean(x, axis=0)     # mean FC over time windows
            sd_fc[s] = np.nanstd(x, axis=0, ddof=1)   # FC variability over time windows
    np.save(os.path.join(CPM_DIR, f"{name}_FC_mean_{ATLAS}.npy"), mean_fc)
    np.save(os.path.join(CPM_DIR, f"{name}_FC_sd_{ATLAS}.npy"), sd_fc)
    n_const = int((sd_fc == 0).sum())              # e.g. an ROI with no signal gives FC fixed at 0
    print(f"{name:12s} {fc.shape} -> {name}_FC_mean_{ATLAS}.npy, {name}_FC_sd_{ATLAS}.npy"
          + (f"   ({n_const} subject-edge values with zero variability)" if n_const else ""))

# Training

In [ ]:
_CORR = {"fast_ok": FAST_CORR}


def _corr_loop(fc_clean, a_clean):
    r = np.empty(fc_clean.shape[1])
    for f in range(fc_clean.shape[1]):
        r[f] = stats.pearsonr(fc_clean[:, f], a_clean)[0]
    return r


def _corr_vec(fc_clean, a_clean, chunk=4096):
    n_feat = fc_clean.shape[1]
    r = np.empty(n_feat)
    for s in range(0, n_feat, chunk):
        X = np.ascontiguousarray(fc_clean[:, s:s + chunk].T)          # (features, TRs)
        r[s:s + chunk] = stats.pearsonr(X, a_clean[np.newaxis, :], axis=1)[0]
    return r


def subject_correlations(fc_s, a_s):
    """Pearson r of every FC feature with arousal for one subject (same valid-TR mask as before)."""
    valid = ~np.isnan(a_s) & ~np.isnan(fc_s).any(axis=1)
    fc_clean, a_clean = fc_s[valid], a_s[valid]
    n_valid = len(a_clean)
    if n_valid < 2:
        return np.full(fc_s.shape[1], np.nan), n_valid
    if _CORR["fast_ok"]:
        r = _corr_vec(fc_clean, a_clean)
        # self-check against the old one-feature-at-a-time pearsonr (bitwise)
        check = np.unique(np.r_[np.linspace(0, len(r) - 1, 64).astype(int),
                                np.where(~np.isfinite(r) | (np.abs(r) == 1))[0][:64]])
        ref = np.array([stats.pearsonr(fc_clean[:, f], a_clean)[0] for f in check], dtype=float)
        if np.array_equal(r[check], ref, equal_nan=True):
            return r, n_valid
        print("  ! vectorised pearsonr is not bit-identical on this SciPy version -> using the exact loop")
        _CORR["fast_ok"] = False
    return _corr_loop(fc_clean, a_clean), n_valid


def _file_sig(path):
    st = os.stat(path)
    return [os.path.abspath(path), st.st_size, st.st_mtime_ns]


def get_r_all(name):
    """(n_subj, n_features) per-subject correlations + n valid TRs per subject (cached)."""
    os.makedirs(CACHE_DIR, exist_ok=True)
    cache_npz = os.path.join(CACHE_DIR, f"{name}_r_all.npz")
    cache_meta = os.path.join(CACHE_DIR, f"{name}_r_all.json")
    sig = {"fc": _file_sig(FC_FILES[name][0]), "arousal": _file_sig(FC_FILES[name][1]), "version": 1}
    if not RECOMPUTE_CACHE and os.path.exists(cache_npz) and os.path.exists(cache_meta):
        with open(cache_meta) as f:
            if json.load(f) == sig:
                d = np.load(cache_npz)
                print(f"{name}: loaded cached correlations")
                return d["r"], d["n_valid"]
    t0 = time.time()
    fc, ar = FC[name], AROUSAL[name]
    r_all = np.full((fc.shape[0], fc.shape[2]), np.nan)
    n_valid = np.zeros(fc.shape[0], dtype=int)
    for s in range(fc.shape[0]):
        r_all[s], n_valid[s] = subject_correlations(np.asarray(fc[s]), ar[s])
    np.savez(cache_npz, r=r_all, n_valid=n_valid)
    with open(cache_meta, "w") as f:
        json.dump(sig, f)
    print(f"{name}: computed correlations for {fc.shape[0]} subjects in {time.time() - t0:.0f}s (cached)")
    return r_all, n_valid


if RUN_PREDICTIONS:
    R_ALL = {name: get_r_all(name) for name in DATASETS}


def r_rows(name, rule):
    """Rows of the correlation matrix exactly as each old function built them.
    'cross'  : cross_modal_prediction / other_to_SR (every subject; pearsonr errors if < 2 TRs)
    'movies' : compute_subject_feature_correlations (NaN row if <= 1 valid TR)
    'loso'   : cross_modal_prediction_loso (NaN row if < 3 valid TRs)"""
    r_all, n_valid = R_ALL[name]
    r = r_all.copy()
    if rule == "cross":
        if np.any(n_valid < 2):
            raise ValueError(f"{name}: a subject has < 2 valid TRs (pearsonr would fail, as before)")
    elif rule == "loso":
        r[n_valid < 3] = np.nan
    return r

# Cross Prediction

In [ ]:
def fit_model(r_matrix, alpha=ALPHA, use_mean=True):
    """One-sample t-test across subjects → significant features + sign weights."""
    _, p = stats.ttest_1samp(r_matrix, popmean=0, axis=0, nan_policy="omit")
    with warnings.catch_warnings():               # NaN rows/subjects are skipped, as in the t-test
        warnings.simplefilter("ignore", RuntimeWarning)
        w = np.nanmean(r_matrix, axis=0) if use_mean else np.nanmedian(r_matrix, axis=0)
    sig = np.where(p < alpha)[0]
    return {"p": p, "weights": w, "sig": sig, "signs": np.sign(w[sig])}


def features_df(model, subject=None):
    pairs = [ROI_PAIRS[i] for i in model["sig"]]
    cols = {} if subject is None else {"Subject": subject}
    cols.update({"ROI1": [p[0] for p in pairs], "ROI2": [p[1] for p in pairs],
                 "p_value": model["p"][model["sig"]], "feature_weight": model["weights"][model["sig"]]})
    return pd.DataFrame(cols)


def predict_cross(fc_target, sig, weights, total_features):
    """cross_modal_prediction / SR_to_other / other_to_SR: (Σ fc·w over w>0 + Σ fc·w over w<0) / n_sig."""
    n_subj, n_trs = fc_target.shape[:2]
    pred = np.full((n_subj, n_trs), np.nan)
    pos, neg = weights > 0, weights < 0
    divisor = total_features if total_features > 0 else 1
    for s in range(n_subj):
        masked = np.asarray(fc_target[s])[:, sig] * weights
        for i in range(n_trs):
            v = masked[i, :]
            pred[s, i] = np.sum(v[pos]) / divisor + np.sum(v[neg]) / divisor
    return pred


def predict_loso_subject(fc_test, weights, total_features):
    """cross_modal_prediction_loso: (Σ raw fc over w>0 − Σ raw fc over w<0) / n_sig, NaN if any NaN in TR."""
    pos, neg = weights > 0, weights < 0
    pred = np.full(fc_test.shape[0], np.nan)
    for t in range(fc_test.shape[0]):
        if np.isnan(fc_test[t]).any():
            continue
        pred[t] = np.sum(fc_test[t][pos]) / total_features - np.sum(fc_test[t][neg]) / total_features
    return pred


MODELS = {}


def source_model(name):
    """Model trained on ALL subjects of one dataset (built once, reused for every target)."""
    if name not in MODELS:
        if name == "Movies":     # Sherlock + FilmFest correlations pooled (old select_and_save_significant_features_v2)
            r = np.vstack([r_rows("Sherlock", "movies"), r_rows("FilmFest", "movies")])
        else:
            r = r_rows(name, "cross")
        MODELS[name] = fit_model(r)
        print(f"Model '{name}': {len(MODELS[name]['sig'])} significant features")
    return MODELS[name]

In [ ]:
STEM = {"FilmFest": "FF", "Sherlock": "Sherlock"}     # 1_Movies_within uses "FF" in file names

ANALYSES = (
    [dict(kind="movies_within", folder="1_Movies_within")]
    + [dict(kind="loso", idx=i, target=t, folder=f"{i}_{t}_within", stem=f"{t}_within")
       for i, t in [(2, "EEG_Rest"), (3, "Pupil_Rest")]]
    + [dict(kind="cross", idx=i, target=t, source=s, folder=f"{i}_{t}_from_{s}", stem=f"{t}_from_{s}")
       for i, t, s in [(4, "EEG_Rest", "Pupil_Rest"), (5, "Pupil_Rest", "EEG_Rest"),
                       (6, "Pupil_Story", "Pupil_Rest"), (7, "Pupil_Rest", "Pupil_Story"),
                       (8, "Pupil_Story", "EEG_Rest"), (9, "EEG_Rest", "Pupil_Story")]]
    + [dict(kind="from_movies", idx=i, target=t, folder=f"{i}_{t}_from_Movies", stem=f"{t}_from_Movies")
       for i, t in [(10, "EEG_Rest"), (12, "Pupil_Rest"), (14, "Pupil_Story")]]
    + [dict(kind="to_movies", idx=i, source=s, folder=f"{i}_Movies_from_{s}", stem=f"Movies_from_{s}")
       for i, s in [(11, "EEG_Rest"), (13, "Pupil_Rest"), (15, "Pupil_Story")]]
    # leave-one-subject-out within the pooled movies (Sherlock + FilmFest minus the test subject)
    + [dict(kind="movies_loso", idx=24, folder="24_Movies_within_LOSO", stem="Movies_LOSO")]
    # leave-one-subject-out within Pupil Story (same procedure as the rest LOSO analyses 2 and 3)
    + [dict(kind="story_loso", idx=25, target="Pupil_Story", folder="25_Pupil_Story_within_LOSO",
            stem="Pupil_Story_within_LOSO")]
)
for a in ANALYSES:
    a["dir"] = os.path.join(CLS_DIR, a["folder"])


def opath(a, suffix, stem=None):
    return os.path.join(a["dir"], f"{stem or a['stem']}_{suffix}")

In [ ]:
def run_movies_within(a):
    # Movies feature list + binarised weights
    m = source_model("Movies")
    save_csv(features_df(m), os.path.join(a["dir"], "Movies_features.csv"))
    save_npy(os.path.join(a["dir"], "Movies_Features_Binarized.npy"), m["signs"])
    # FF ← Sherlock and Sherlock ← FF
    for target, source in [("FilmFest", "Sherlock"), ("Sherlock", "FilmFest")]:
        ms = source_model(source)
        stem = f"{STEM[target]}_from_{STEM[source]}"
        save_npy(opath(a, "predicted.npy", stem), predict_cross(FC[target], ms["sig"], ms["signs"], len(ms["sig"])))
        save_npy(opath(a, "observed.npy", stem), AROUSAL[target])
        save_csv(features_df(ms), opath(a, "features.csv", stem))


def run_loso(a):
    name = a["target"]
    fc, ar = FC[name], AROUSAL[name]
    r_loso = r_rows(name, "loso")
    n_subj, n_trs = fc.shape[:2]
    pred = np.full((n_subj, n_trs), np.nan)
    logs = []
    for test in range(n_subj):
        train = [i for i in range(n_subj) if i != test]
        m = fit_model(r_loso[train])                   # rows of the cached matrix = same as retraining
        if len(m["sig"]) == 0:
            print(f"  No significant features for subject {test + 1}. Skipping prediction.")
            continue
        logs.append(features_df(m, subject=test + 1))
        pred[test] = predict_loso_subject(np.asarray(fc[test])[:, m["sig"]], m["signs"], len(m["sig"]))
    print(f"  LOSO {name}: {n_subj} folds done")
    save_npy(opath(a, "predicted.npy"), pred)
    save_npy(opath(a, "observed.npy"), ar.copy())
    save_csv(pd.concat(logs, ignore_index=True), opath(a, "features.csv"))


def run_cross(a):
    m = source_model(a["source"])
    t = a["target"]
    save_npy(opath(a, "predicted.npy"), predict_cross(FC[t], m["sig"], m["signs"], len(m["sig"])))
    save_npy(opath(a, "observed.npy"), AROUSAL[t])
    save_csv(features_df(m), opath(a, "features.csv"))


def run_from_movies(a):
    m = source_model("Movies")
    t = a["target"]
    save_npy(opath(a, "predicted.npy"), predict_cross(FC[t], m["sig"], m["signs"], len(m["sig"])))
    save_npy(opath(a, "observed.npy"), AROUSAL[t])


def run_to_movies(a):
    m = source_model(a["source"])
    s = a["source"]
    for target in ["FilmFest", "Sherlock"]:
        stem = f"{target}_from_{s}"
        save_npy(opath(a, "predicted.npy", stem), predict_cross(FC[target], m["sig"], m["signs"], len(m["sig"])))
        save_npy(opath(a, "observed.npy", stem), AROUSAL[target])
    save_csv(features_df(m), opath(a, "features.csv"))


def run_movies_loso(a):
    """Movies LOSO: for every Sherlock and FilmFest subject, fit the Movies model on the other movie subjects
    (pooled correlation rows, Sherlock first, exactly as the Movies model) and predict the held-out subject
    with the same formula as the other movie predictions (predict_cross)."""
    r_pool = np.vstack([r_rows("Sherlock", "movies"), r_rows("FilmFest", "movies")])
    owners = ([("Sherlock", i) for i in range(FC["Sherlock"].shape[0])]
              + [("FilmFest", i) for i in range(FC["FilmFest"].shape[0])])
    assert len(owners) == r_pool.shape[0]
    pred = {d: np.full(FC[d].shape[:2], np.nan) for d in ("FilmFest", "Sherlock")}
    logs = []
    for k, (d, i) in enumerate(owners):
        m = fit_model(np.delete(r_pool, k, axis=0))          # = the Movies model without this subject's row
        if len(m["sig"]) == 0:
            print(f"  No significant features without {d} subject {i + 1}. Skipping prediction.")
            continue
        df = features_df(m, subject=i + 1)
        df.insert(0, "Test_dataset", d)
        logs.append(df)
        pred[d][i] = predict_cross(FC[d][i:i + 1], m["sig"], m["signs"], len(m["sig"]))[0]
    print(f"  Movies LOSO: {len(owners)} folds done")
    for d in ("FilmFest", "Sherlock"):
        save_npy(opath(a, "predicted.npy", f"{d}_from_Movies_LOSO"), pred[d])
        save_npy(opath(a, "observed.npy", f"{d}_from_Movies_LOSO"), AROUSAL[d])
    save_csv(pd.concat(logs, ignore_index=True) if logs else
             pd.DataFrame(columns=["Test_dataset", "Subject", "ROI1", "ROI2", "p_value", "feature_weight"]),
             opath(a, "features.csv"))


RUNNERS = {"movies_within": run_movies_within, "loso": run_loso, "cross": run_cross,
           "from_movies": run_from_movies, "to_movies": run_to_movies, "movies_loso": run_movies_loso,
           "story_loso": run_loso}

if RUN_PREDICTIONS:
    for a in ANALYSES:
        t0 = time.time()
        RUNNERS[a["kind"]](a)
        print(f"✅ {a['folder']} ({time.time() - t0:.1f}s)")

# Permutation Test

In [ ]:
def fisher_z(r):
    r = np.clip(r, -0.999999, 0.999999)
    return 0.5 * (np.log1p(r) - np.log1p(-r))


def fisher_z_inv(z):
    return (np.exp(2 * z) - 1) / (np.exp(2 * z) + 1)


def compute_observed_value(predicted_arousal, observed_arousal, stat="median", corr_type="pearson", return_all_r=False):
    if predicted_arousal.shape[0] != observed_arousal.shape[0]:
        min_subjects = min(predicted_arousal.shape[0], observed_arousal.shape[0])
        print(f"Warning: Mismatched subject counts! Using first {min_subjects} subjects.")
        predicted_arousal = predicted_arousal[:min_subjects]
        observed_arousal = observed_arousal[:min_subjects]
    observed_correlations = []
    for subj in range(predicted_arousal.shape[0]):
        valid_mask = ~np.isnan(predicted_arousal[subj]) & ~np.isnan(observed_arousal[subj])
        if np.sum(valid_mask) > 1:
            if corr_type == "pearson":
                corr, _ = pearsonr(predicted_arousal[subj][valid_mask], observed_arousal[subj][valid_mask])
            elif corr_type == "spearman":
                corr, _ = spearmanr(predicted_arousal[subj][valid_mask], observed_arousal[subj][valid_mask])
            else:
                raise ValueError("Invalid correlation type. Choose 'pearson' or 'spearman'.")
            observed_correlations.append(corr)
    if len(observed_correlations) == 0:
        raise ValueError("No valid correlations computed. Check your data for NaN values.")
    observed_correlations = np.array(observed_correlations)
    if return_all_r:
        return observed_correlations
    z_vals = fisher_z(observed_correlations)
    z_stat = np.median(z_vals) if stat == "median" else np.mean(z_vals)
    return fisher_z_inv(z_stat)


def phase_randomize(data, random_state=None):
    random_state = check_random_state(random_state)
    data = np.array(data)
    fft_data = fft(data, axis=0)
    pos_freq, neg_freq = _phase_freqs(data.shape[0])
    if len(data.shape) == 1:
        phase_shifts = random_state.uniform(0, 2 * np.pi, size=(len(pos_freq)))
        fft_data[pos_freq] *= np.exp(1j * phase_shifts)
        fft_data[neg_freq] *= np.exp(-1j * phase_shifts)
    else:
        phase_shifts = random_state.uniform(0, 2 * np.pi, size=(len(pos_freq), data.shape[1]))
        fft_data[pos_freq, :] *= np.exp(1j * phase_shifts)
        fft_data[neg_freq, :] *= np.exp(-1j * phase_shifts)
    return np.real(ifft(fft_data, axis=0))


def _phase_freqs(n):
    if n % 2 == 0:
        return np.arange(1, n // 2), np.arange(n - 1, n // 2, -1)
    return np.arange(1, (n - 1) // 2 + 1), np.arange(n - 1, (n - 1) // 2, -1)


# ---- batched null: same random draws in the same order as phase_randomize -----
def _phase_randomize_batch(obs, phases):
    """obs (n_subj, T); phases (B, n_subj, n_pos) → (B, n_subj, T) surrogates."""
    pos_freq, neg_freq = _phase_freqs(obs.shape[-1])
    if len(pos_freq) == 0:
        return np.real(ifft(np.broadcast_to(fft(obs, axis=-1), phases.shape[:2] + obs.shape[-1:]), axis=-1))
    pos = slice(pos_freq[0], pos_freq[-1] + 1)               # same bins as the index arrays, as slices
    neg = slice(neg_freq[0], neg_freq[-1] - 1, -1)
    F = np.broadcast_to(fft(obs, axis=-1), phases.shape[:2] + obs.shape[-1:]).copy()
    E = np.exp(1j * phases)
    F[..., pos] *= E
    F[..., neg] *= E.conj()                                  # == np.exp(-1j * phases), bit for bit
    return np.real(ifft(F, axis=-1))


def _draw_phases(rs, B, groups):
    """groups = [(n_subj, T), ...] drawn per permutation in this order (like the old loops)."""
    sizes = [n * len(_phase_freqs(T)[0]) for n, T in groups]
    flat = rs.uniform(0, 2 * np.pi, size=B * sum(sizes)).reshape(B, sum(sizes))
    out_, start = [], 0
    for (n, T), sz in zip(groups, sizes):
        out_.append(flat[:, start:start + sz].reshape(B, n, sz // n))
        start += sz
    return out_


def _rowwise_r(pred, surr, masked):
    """Pearson r per (perm, subject). masked=True mimics compute_observed_value's NaN masking."""
    P = np.broadcast_to(pred, surr.shape)
    if masked and not (np.isnan(pred).any() or np.isnan(surr).any()):
        masked = False                                       # nothing to mask -> identical, faster maths
    if masked:
        valid = ~np.isnan(P) & ~np.isnan(surr)
    else:
        valid = np.ones(surr.shape, dtype=bool)
    n = valid.sum(-1)
    with np.errstate(invalid="ignore", divide="ignore"):
        x = np.where(valid, P, 0.0) if masked else P
        y = np.where(valid, surr, 0.0) if masked else surr
        xm = x - x.sum(-1, keepdims=True) / n[..., None]
        ym = y - y.sum(-1, keepdims=True) / n[..., None]
        if masked:
            xm, ym = np.where(valid, xm, 0.0), np.where(valid, ym, 0.0)
        r = (xm * ym).sum(-1) / np.sqrt((xm * xm).sum(-1) * (ym * ym).sum(-1))
    return np.clip(r, -1.0, 1.0), n


def _group_mean_r(r, include):
    """Fisher-z mean over included subjects → back to r (NaN propagates as in np.mean)."""
    z = np.where(include, fisher_z(r), 0.0)
    k = include.sum(-1)
    if np.any(k == 0):
        raise ValueError("No valid correlations computed. Check your data for NaN values.")
    return fisher_z_inv(z.sum(-1) / k)


def _batches(n_perm, per_perm_elems, budget=4_000_000):
    B = max(1, int(budget // max(per_perm_elems, 1)))
    for s in range(0, n_perm, B):
        yield min(B, n_perm - s)


def fast_null_masked(predicted, observed, n_perm):
    rs = check_random_state(None)                  # the global numpy RandomState, like phase_randomize
    n_min = min(predicted.shape[0], observed.shape[0])
    null = []
    for B in _batches(n_perm, observed.size):
        (ph,) = _draw_phases(rs, B, [observed.shape])
        surr = _phase_randomize_batch(observed, ph)[:, :n_min]
        r, n = _rowwise_r(predicted[:n_min], surr, masked=True)
        null.append(_group_mean_r(r, n > 1))
    return np.concatenate(null)


def fast_null_two(pred_a, obs_a, pred_b, obs_b, n_perm, order="ab"):
    """Unmasked r for two movie datasets; draws a then b per permutation.
    order = order in which the r-values are concatenated."""
    rs = check_random_state(None)
    null = []
    for B in _batches(n_perm, obs_a.size + obs_b.size):
        ph_a, ph_b = _draw_phases(rs, B, [obs_a.shape, obs_b.shape])
        ra, _ = _rowwise_r(pred_a, _phase_randomize_batch(obs_a, ph_a), masked=False)
        rb, _ = _rowwise_r(pred_b, _phase_randomize_batch(obs_b, ph_b), masked=False)
        r = np.concatenate([ra, rb] if order == "ab" else [rb, ra], axis=1)
        null.append(_group_mean_r(r, np.ones(r.shape, dtype=bool)))
    return np.concatenate(null)


def save_perm_results(save_path, null_distributions, observed_rvals, p_value, observed_value,
                      stat="mean", corr_type="pearson", subject_csv=True):
    base = os.path.splitext(save_path)[0]
    save_npy(base + "_null_distribution.npy", null_distributions)
    save_npy(base + "_observed_rvals.npy", observed_rvals)
    save_txt(base + "_pval.txt", f"{p_value:.6f}")
    if subject_csv:
        save_csv(pd.DataFrame({"subject": np.arange(len(observed_rvals)), f"{corr_type}_r": observed_rvals}),
                 base + "_subject_rvals.csv")
    save_txt(base + f"_{stat}_observed_r.txt", f"{observed_value:.6f}")
    print(f"Saved results to {base}_*.npy/.txt/.csv")


def plot_null(null_distributions, observed_value, null_type, stat, corr_type):
    if not SHOW_PERM_PLOTS:
        return
    null_distributions = np.asarray(null_distributions)[np.isfinite(null_distributions)]
    if null_distributions.size == 0:
        print("  (null distribution is all NaN — no plot; did this model select any features?)")
        return
    plt.figure(figsize=(8, 6))
    plt.hist(null_distributions, bins=50, color="gray", alpha=0.7, density=True, label=f"Null Distribution ({null_type})")
    plt.axvline(observed_value, color="red", linestyle="dotted", linewidth=2,
                label=f"Observed {stat.capitalize()} {corr_type.capitalize()} Correlation")
    plt.xlabel(f"{stat.capitalize()} {corr_type.capitalize()} Correlation")
    plt.ylabel("Density")
    plt.title(f"Permutation Test ({null_type.replace('_', ' ').title()})")
    plt.grid(True)
    plt.legend()
    plt.show()


def permutation_test(predicted_arousal, observed_arousal, null_type="phase_randomization", stat="median",
                     n_permutations=10000, save_path=None, corr_type="pearson"):
    observed_rvals = compute_observed_value(predicted_arousal, observed_arousal, stat, corr_type, return_all_r=True)
    z_vals = fisher_z(observed_rvals)
    observed_value = fisher_z_inv(np.median(z_vals) if stat == "median" else np.mean(z_vals))
    print(f"Observed {stat} {corr_type} correlation: {observed_value:.4f}")

    if FAST_NULL and null_type == "phase_randomization" and corr_type == "pearson" and stat == "mean":
        null_distributions = fast_null_masked(predicted_arousal, observed_arousal, n_permutations)
    else:                                               # original loop (spearman / median / between_subject)
        null_distributions = []
        for _ in range(n_permutations):
            if null_type == "phase_randomization":
                shuffled = np.array([phase_randomize(a) for a in observed_arousal])
            elif null_type == "between_subject":
                shuffled = observed_arousal[np.random.permutation(observed_arousal.shape[0])]
            else:
                raise ValueError("Invalid null_type. Choose 'phase_randomization' or 'between_subject'.")
            null_distributions.append(compute_observed_value(predicted_arousal, shuffled, stat, corr_type))
        null_distributions = np.array(null_distributions)

    p_value = (np.sum(null_distributions >= observed_value) + 1) / (n_permutations + 1)
    print(f"P-value: {p_value:.4f}")
    if save_path:
        save_perm_results(save_path, null_distributions, observed_rvals, p_value, observed_value, stat, corr_type)
    plot_null(null_distributions, observed_value, null_type, stat, corr_type)
    return observed_value, null_distributions, p_value


def _movie_rvals(pred_a, obs_a, pred_b, obs_b):
    return np.array([pearsonr(pred_a[i], obs_a[i])[0] for i in range(pred_a.shape[0])]
                    + [pearsonr(pred_b[i], obs_b[i])[0] for i in range(pred_b.shape[0])])


def _slow_null_two(pred_ff, ff_obs, pred_s, s_obs, n_perm, order):
    null = []
    for _ in range(n_perm):
        sh_ff = np.array([phase_randomize(a) for a in ff_obs])
        sh_s = np.array([phase_randomize(a) for a in s_obs])
        r_ff = [pearsonr(pred_ff[i], sh_ff[i])[0] for i in range(sh_ff.shape[0])]
        r_s = [pearsonr(pred_s[i], sh_s[i])[0] for i in range(sh_s.shape[0])]
        r = np.array(r_ff + r_s if order == "ab" else r_s + r_ff)
        null.append(fisher_z_inv(np.mean(fisher_z(r))))
    return np.array(null)


def permutation_test_toSR(toFF_predicted, FF_observed, toS_predicted, S_observed, null_type="phase_randomization",
                          stat="median", n_permutations=10000, save_path=None, corr_type="pearson"):
    if null_type != "phase_randomization":
        raise ValueError("Invalid null_type. Choose 'phase_randomization'.")
    observed_rvals = _movie_rvals(toFF_predicted, FF_observed, toS_predicted, S_observed)
    z_vals = fisher_z(observed_rvals)
    observed_value = fisher_z_inv(np.median(z_vals) if stat == "median" else np.mean(z_vals))
    print(f"Observed {stat} {corr_type} correlation: {observed_value:.4f}")
    if stat != "mean":
        raise NotImplementedError("only stat='mean' is used in this pipeline")
    if FAST_NULL:
        null_distributions = fast_null_two(toFF_predicted, FF_observed, toS_predicted, S_observed, n_permutations, "ab")
    else:
        null_distributions = _slow_null_two(toFF_predicted, FF_observed, toS_predicted, S_observed, n_permutations, "ab")
    p_value = (np.sum(null_distributions >= observed_value) + 1) / (n_permutations + 1)
    print(f"P-value: {p_value:.4f}")
    if save_path:
        save_perm_results(save_path, null_distributions, observed_rvals, p_value, observed_value, stat, corr_type)
    plot_null(null_distributions, observed_value, null_type, stat, corr_type)
    return observed_value, null_distributions, p_value

In [ ]:
BY_KIND = {k: [a for a in ANALYSES if a["kind"] == k] for k in RUNNERS}

if RUN_PERMUTATIONS:
    if SEED is not None:
        np.random.seed(SEED)

    # within-dataset, cross-dataset and from-movies tests (old folders 2–10, 12, 14; same order)
    for a in sorted(BY_KIND["loso"] + BY_KIND["cross"] + BY_KIND["from_movies"], key=lambda a: a["idx"]):
        pred_path, obs_path = opath(a, "predicted.npy"), opath(a, "observed.npy")
        if not os.path.exists(pred_path) or not os.path.exists(obs_path):
            print(f"Skipping {a['folder']}: Missing one or both files.")
            continue
        print(f"Running {a['folder']}")
        permutation_test(np.load(pred_path), np.load(obs_path), null_type="phase_randomization", stat="mean",
                         n_permutations=N_PERM, corr_type="pearson", save_path=os.path.join(a["dir"], PERM_BASE))

    # rest/story → movies (old folders 11, 13, 15)
    mw = BY_KIND["movies_within"][0]
    FF_observed = np.load(opath(mw, "observed.npy", "FF_from_Sherlock"))
    Sher_observed = np.load(opath(mw, "observed.npy", "Sherlock_from_FF"))
    for a in BY_KIND["to_movies"]:
        s = a["source"]
        print(f"Running {a['folder']}")
        permutation_test_toSR(np.load(opath(a, "predicted.npy", f"FilmFest_from_{s}")), FF_observed,
                              np.load(opath(a, "predicted.npy", f"Sherlock_from_{s}")), Sher_observed,
                              null_type="phase_randomization", stat="mean", n_permutations=N_PERM,
                              save_path=os.path.join(a["dir"], PERM_BASE), corr_type="pearson")

    # movies within (Sherlock←FF and FF←Sherlock pooled) — saved next to the other Movies-within outputs
    FF_Sher_predicted = np.load(opath(mw, "predicted.npy", "Sherlock_from_FF"))
    Sher_FF_predicted = np.load(opath(mw, "predicted.npy", "FF_from_Sherlock"))
    SR_rvals = _movie_rvals(FF_Sher_predicted, Sher_observed, Sher_FF_predicted, FF_observed)
    SR_avg_r = fisher_z_inv(np.mean(fisher_z(SR_rvals)))
    # old loop drew FF surrogates first, then Sherlock, but listed Sherlock r-values first
    if FAST_NULL:
        fake_r = fast_null_two(Sher_FF_predicted, FF_observed, FF_Sher_predicted, Sher_observed, N_PERM, order="ba")
    else:
        fake_r = _slow_null_two(Sher_FF_predicted, FF_observed, FF_Sher_predicted, Sher_observed, N_PERM, order="ba")
    p_value = (np.sum(fake_r >= SR_avg_r) + 1) / (N_PERM + 1)
    print(f"SR from SR avg r: {SR_avg_r:.4f}, p-value: {p_value:.4f}")
    save_perm_results(os.path.join(mw["dir"], PERM_BASE), fake_r, SR_rvals, p_value, SR_avg_r,
                      stat="mean", subject_csv=False)

    # movies LOSO (24): FilmFest + Sherlock pooled, same test as the rest/story -> movies analyses
    for a in BY_KIND["movies_loso"]:
        print(f"Running {a['folder']}")
        permutation_test_toSR(np.load(opath(a, "predicted.npy", "FilmFest_from_Movies_LOSO")), FF_observed,
                              np.load(opath(a, "predicted.npy", "Sherlock_from_Movies_LOSO")), Sher_observed,
                              null_type="phase_randomization", stat="mean", n_permutations=N_PERM,
                              save_path=os.path.join(a["dir"], PERM_BASE), corr_type="pearson")

    # Pupil Story LOSO (25): same test as the within-dataset rest LOSO analyses (2, 3)
    for a in BY_KIND["story_loso"]:
        pred_path, obs_path = opath(a, "predicted.npy"), opath(a, "observed.npy")
        if not os.path.exists(pred_path) or not os.path.exists(obs_path):
            print(f"Skipping {a['folder']}: Missing one or both files.")
            continue
        print(f"Running {a['folder']}")
        permutation_test(np.load(pred_path), np.load(obs_path), null_type="phase_randomization", stat="mean",
                         n_permutations=N_PERM, corr_type="pearson", save_path=os.path.join(a["dir"], PERM_BASE))

# Sleep Classification

In [ ]:
import gc
from nilearn.image import clean_img
try:
    from nilearn.maskers import NiftiLabelsMasker
except ImportError:
    from nilearn.input_data import NiftiLabelsMasker

RUN_SLEEP        = True
REBUILD_SLEEP_FC = False   # True = recompute the per-run sleep FC files even if they exist

PROJECT       = os.path.dirname(RESULTS)                                   # ".."
ATLASES       = {"Shen": "3_atlases/Shen/shen_2mm_268_parcellation.nii.gz",                     # same as in a
                 "Schaefer": "3_atlases/Schaefer/Schaefer200_17N_plus_8ROIs_208Parcels.nii.gz"}
ATLAS_PATH    = os.path.join(PROJECT, ATLASES[ATLAS])
REST_DIR      = os.path.join(PROJECT, "1_data", "1_EEG_Rest_fMRI")
SLEEP_DIR     = os.path.join(PROJECT, "1_data", "1_EEG_Sleep_fMRI")
STAGE_DIR     = os.path.join(SLEEP_DIR, "wakestage")
SLEEP_FC_DIR  = res("1_arousal", "1_EEG_Sleep", f"sleep_fc_{ATLAS}")      # per-run sleep FC (atlas-specific)

N_REST_EPOCHS = 40         # staging rows before the sleep runs start
WAKEMAP = {"W": 4, "W (uncertain)": 4, "1": 3, "1 (uncertain)": 3,
           "2": 2, "2 (uncertain)": 2, "3": 1, "3 (uncertain)": 1}
# Never tested: external exclusions + sub-02/sub-21 (no sleep runs)
EXCLUDE = ["sub-01", "sub-02", "sub-03", "sub-09", "sub-15", "sub-16",
           "sub-21", "sub-24", "sub-27", "sub-30", "sub-32"]
DROP_EXTERNAL_FROM_EEG_TRAINING = True      # also drop EXCLUDE subjects from the EEG Rest training folds

SLEEP_TR = 2.1
SLEEP_EPOCH_START = np.round(np.array(range(0, 871, 30)) / SLEEP_TR)    # 30 epochs of 30 s per run
SLEEP_EPOCH_END   = np.round(np.array(range(30, 901, 30)) / SLEEP_TR)
SLEEP_CONFOUNDS = ["global_signal", "white_matter", "csf",
                   "trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]


def predict_from_model(fc, sig, w):
    """(FC * weight), avg_pos + avg_neg, divided by total features (same as the cross-dataset models)."""
    fc_sig = np.asarray(fc)[:, sig]
    total = len(sig) if len(sig) > 0 else 1
    masked = fc_sig * w
    pred = masked[:, w > 0].sum(axis=1) / total + masked[:, w < 0].sum(axis=1) / total
    check = fc_sig[:, w > 0].sum(axis=1) / total - fc_sig[:, w < 0].sum(axis=1) / total
    assert np.allclose(pred, check, equal_nan=True), "Sign error: negative edges double-flipped"
    return pred


def direction_check(arousal_array, fc_array, sig, w, label):
    """Model applied to its own training data should correlate positively with arousal."""
    rs = []
    for s in range(arousal_array.shape[0]):
        pred = predict_from_model(fc_array[s], sig, w)
        a = arousal_array[s]
        ok = ~np.isnan(pred) & ~np.isnan(a)
        if ok.sum() > 2 and np.ptp(a[ok]) > 0:
            rs.append(pearsonr(pred[ok], a[ok])[0])
    m = fisher_z_inv(np.mean(fisher_z(np.array(rs))))
    print(f"Direction check ({label}, in-sample): mean r = {m:.3f}  -> should be positive")
    if m <= 0:
        print("  ⚠️ Model does not predict its own training data positively. Check signs.")
    return m


def sleep_feature_df(model, **extra):
    df = features_df(model)
    for k, v in extra.items():
        df.insert(0, k, v)
    return df


def predict_test_set(test_set, model_for):
    pred, obs = {}, {}
    for subj, d in test_set.items():
        model = model_for(subj)
        if model is None:
            continue
        sig, w = model
        if len(sig) == 0:
            print(f"{subj}: no significant features, skipped")
            continue
        p_ = predict_from_model(d["fc"], sig, w)
        o_ = d["stage"]
        ok = ~np.isnan(p_) & ~np.isnan(o_)
        p_, o_ = p_[ok], o_[ok]
        if len(o_) < 3 or np.ptp(o_) == 0 or np.ptp(p_) == 0:
            print(f"{subj}: dropped (n = {len(o_)} or constant stage series)")
            continue
        pred[subj], obs[subj] = p_, o_
        print(f"{subj}: r = {pearsonr(p_, o_)[0]:.4f}  ({len(o_)} epochs)")
    return pred, obs


def fast_null_subjects(pred_list, obs_list, n_perm):
    """Batched version of the per-subject phase-randomisation loop (same draws, same order)."""
    rs = check_random_state(None)
    groups = [(1, len(o)) for o in obs_list]
    null = []
    for B in _batches(n_perm, sum(len(o) for o in obs_list)):
        phases = _draw_phases(rs, B, groups)
        r = np.concatenate([_rowwise_r(p[np.newaxis, :], _phase_randomize_batch(o[np.newaxis, :], ph), masked=False)[0]
                            for p, o, ph in zip(pred_list, obs_list, phases)], axis=1)
        null.append(_group_mean_r(r, np.ones(r.shape, dtype=bool)))
    return np.concatenate(null)


def permutation_test_sleep(pred, obs, n_perm=N_PERM):
    """Phase-randomize observed stages, Fisher-z mean, p = (count + 1) / (n + 1)."""
    subjects = list(pred)
    rvals = np.array([pearsonr(pred[s], obs[s])[0] for s in subjects])
    observed = fisher_z_inv(np.mean(fisher_z(rvals)))
    if FAST_NULL:
        null = fast_null_subjects([pred[s] for s in subjects], [obs[s] for s in subjects], n_perm)
    else:
        null = np.empty(n_perm)
        for i in range(n_perm):
            rr = np.array([pearsonr(pred[s], phase_randomize(obs[s]))[0] for s in subjects])
            null[i] = fisher_z_inv(np.mean(fisher_z(rr)))
    p = (np.sum(null >= observed) + 1) / (n_perm + 1)
    print(f"\nMean r = {observed:.4f}, p = {p:.4f}, n = {len(rvals)}")
    return observed, rvals, null, p


def save_sleep_results(folder, stem, pred, obs, mean_r, rvals, null, p, features):
    out_dir = os.path.join(CLS_DIR, folder)
    os.makedirs(out_dir, exist_ok=True)
    b = os.path.join(out_dir, stem)
    np.save(b + "_predicted.npy", pred)
    np.save(b + "_observed.npy", obs)
    np.save(b + "_corrs.npy", rvals)
    np.save(b + "_iters.npy", null)
    np.save(b + "_r.npy", mean_r)
    np.save(b + "_p.npy", p)
    features.to_csv(b + "_features.csv", index=False)
    pd.DataFrame({"subject_id": list(pred), "pearson_r": rvals}).to_csv(b + "_subject_rvals.csv", index=False)
    print(f"Saved to {b}_*")

In [ ]:
if RUN_SLEEP:
    os.makedirs(SLEEP_FC_DIR, exist_ok=True)
    sleep_files = sorted(f for f in os.listdir(SLEEP_DIR) if f.endswith(".nii.gz"))
    n_done = 0
    for file in sleep_files:
        subj = file.split("_")[0]
        if subj in EXCLUDE:
            continue
        run = int("".join(c for c in file.split("_")[2] if c.isdigit()))
        out_path = os.path.join(SLEEP_FC_DIR, f"{subj}_run-{run:02d}_sleep_fc.npy")
        if os.path.exists(out_path) and not REBUILD_SLEEP_FC:
            n_done += 1
            continue
        confound_path = os.path.join(SLEEP_DIR, f"{subj}_task-sleep_run-{run}_desc-confounds_timeseries.tsv")
        if not os.path.exists(confound_path):
            print(f"{subj} run-{run}  |  missing confounds, skipped")
            continue

        confounds = pd.read_csv(confound_path, sep="\t")
        max_t = confounds[["trans_x", "trans_y", "trans_z"]].abs().max().max()
        max_r = confounds[["rot_x", "rot_y", "rot_z"]].abs().max().max()
        flags = []
        if max_t > 3.0:
            flags.append(f"⚠️ MOTION outlier (max translation {max_t:.2f} mm)")
        if max_r > 3 * (np.pi / 180):
            flags.append(f"⚠️ ROTATION outlier (max rotation {np.degrees(max_r):.2f}°)")
        print(f"Sleep FC  {subj} run-{run}" + ("  |  " + "  |  ".join(flags) if flags else ""))

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", FutureWarning)
            warnings.simplefilter("ignore", DeprecationWarning)
            cleaned_img = clean_img(os.path.join(SLEEP_DIR, file), detrend=False, standardize=True,
                                    confounds=confounds[SLEEP_CONFOUNDS].copy())
            time_series = NiftiLabelsMasker(labels_img=ATLAS_PATH).fit_transform(X=cleaned_img)
        n_TRs, n_regions = time_series.shape
        n_features = (n_regions * (n_regions - 1)) // 2
        iu = np.triu_indices(n_regions, k=1)

        vecs = []
        for i in range(len(SLEEP_EPOCH_START)):
            start, end = int(SLEEP_EPOCH_START[i]), min(int(SLEEP_EPOCH_END[i]), n_TRs)
            ts = time_series[start:end]
            if ts.shape[0] < 2:
                vecs.append(np.full(n_features, np.nan))
                continue
            cov = np.cov(ts.T)
            std = np.sqrt(np.diag(cov))
            std[std == 0] = 1e-6
            vecs.append((cov / np.outer(std, std))[iu])
        np.save(out_path, np.stack(vecs))
        del cleaned_img, time_series
        gc.collect()
    if n_done:
        print(f"{n_done} sleep runs already had FC files in {SLEEP_FC_DIR} (set REBUILD_SLEEP_FC = True to redo)")

In [ ]:
if RUN_SLEEP:
    if "R_ALL" not in globals():
        R_ALL = {name: get_r_all(name) for name in DATASETS}

    # ---- test set: sleep FC + stage per subject ----
    nii_runs = {}
    for f in os.listdir(SLEEP_DIR):
        if f.endswith(".nii.gz"):
            nii_runs.setdefault(f.split("_")[0], set()).add(int("".join(c for c in f.split("_")[2] if c.isdigit())))
    test_set = {}
    for subj in sorted(nii_runs):
        if subj in EXCLUDE:
            continue
        runs = sorted(nii_runs[subj])
        fc_paths = [os.path.join(SLEEP_FC_DIR, f"{subj}_run-{r:02d}_sleep_fc.npy") for r in runs]
        if not all(os.path.exists(p) for p in fc_paths):
            print(f"{subj}: missing FC for some runs, skipped")
            continue
        tsv = os.path.join(STAGE_DIR, f"{subj}-sleep-stage.tsv")
        if not os.path.exists(tsv):
            print(f"{subj}: no stage file, skipped")
            continue
        raw = pd.read_csv(tsv, sep="\t")["30-sec_epoch_sleep_stage"]
        unmapped = sorted(set(raw.dropna().astype(str)) - set(WAKEMAP))
        sleep_stage = raw.map(WAKEMAP).values.astype(float)[N_REST_EPOCHS:]
        fc = np.concatenate([np.load(p) for p in fc_paths], axis=0)
        if fc.shape[0] != len(sleep_stage):
            print(f"{subj}: FC has {fc.shape[0]} epochs but staging has {len(sleep_stage)} sleep epochs, skipped")
            continue
        test_set[subj] = {"fc": fc, "stage": sleep_stage}
        if unmapped:
            print(f"{subj}: unmapped labels → NaN: {unmapped}")
    print(f"Sleep test subjects ({len(test_set)}): {list(test_set)}")
    if SEED is not None:
        np.random.seed(SEED)

    # ---- 16 / 17: Pupil Rest and Pupil Story models (trained on all subjects; same models as tests 4–9) ----
    for name, folder in [("Pupil_Rest", "16_Wakestage_from_Pupil_Rest"), ("Pupil_Story", "17_Wakestage_from_Pupil_Story")]:
        print(f"\n=== {folder} ===")
        m = source_model(name)
        direction_check(AROUSAL[name], FC[name], m["sig"], m["signs"], name.replace("_", " "))
        pred, obs = predict_test_set(test_set, lambda s, m=m: (m["sig"], m["signs"]))
        mean_r, rvals, null, p = permutation_test_sleep(pred, obs)
        save_sleep_results(folder, f"Wakestage_Sleep_from_{name}", pred, obs, mean_r, rvals, null, p, features_df(m))

    # ---- 18: Movies model (Sherlock + FilmFest) ----
    print("\n=== 18_Wakestage_from_Movies ===")
    m = source_model("Movies")
    pred, obs = predict_test_set(test_set, lambda s: (m["sig"], m["signs"]))
    mean_r, rvals, null, p = permutation_test_sleep(pred, obs)
    save_sleep_results("18_Wakestage_from_Movies", "Wakestage_Sleep_from_Movies", pred, obs, mean_r, rvals, null, p,
                       features_df(m))

    # ---- 19: EEG Rest, leaving the test subject out of training ----
    print("\n=== 19_Wakestage_from_EEG_Rest ===")
    rest_subjects = sorted({f.split("_")[0] for f in os.listdir(REST_DIR) if f.endswith(".nii.gz")})
    removed = DATASETS["EEG_Rest"]["remove"]
    eeg_subjects = [s for i, s in enumerate(rest_subjects) if i not in removed]
    assert [rest_subjects[i] for i in removed] == ["sub-03", "sub-09"], \
        f"EEG rows {removed} are {[rest_subjects[i] for i in removed]}, expected sub-03/sub-09. Check the rest folder."
    assert len(eeg_subjects) == AROUSAL["EEG_Rest"].shape[0]
    r_full = r_rows("EEG_Rest", "loso")                 # per-subject r, NaN rows for < 3 valid TRs
    base_train = [i for i, s in enumerate(eeg_subjects)
                  if not (DROP_EXTERNAL_FROM_EEG_TRAINING and s in EXCLUDE)]
    eeg_feature_logs = []

    def eeg_loso_model(subj):
        if subj not in eeg_subjects:
            print(f"{subj}: not in EEG Rest data, skipped")
            return None
        left_out = eeg_subjects.index(subj)
        train_idx = [i for i in base_train if i != left_out]
        mm = fit_model(r_full[train_idx])
        print(f"{subj}: left out of EEG training ({len(train_idx)} training subjects, {len(mm['sig'])} features)")
        eeg_feature_logs.append(sleep_feature_df(mm, EEG_Subject_Left_Out=subj, N_EEG_Training_Subjects=len(train_idx)))
        return mm["sig"], mm["signs"]

    pred, obs = predict_test_set(test_set, eeg_loso_model)
    mean_r, rvals, null, p = permutation_test_sleep(pred, obs)
    save_sleep_results("19_Wakestage_from_EEG_Rest", "Wakestage_Sleep_from_EEG_Rest", pred, obs, mean_r, rvals, null, p,
                       pd.concat(eeg_feature_logs, ignore_index=True))

# Export CPM Inputs

In [ ]:
if RUN_CPM_EXPORT:
    if "R_ALL" not in globals():
        R_ALL = {name: get_r_all(name) for name in DATASETS}
    os.makedirs(CPM_DIR, exist_ok=True)
    pd.DataFrame({"feature": np.arange(N_FEATURES), "ROI1": [p[0] for p in ROI_PAIRS],
                  "ROI2": [p[1] for p in ROI_PAIRS]}).to_csv(
        os.path.join(CPM_DIR, f"feature_roi_pairs_{ATLAS}.csv"), index=False)
    for name in list(DATASETS) + ["Movies"]:
        if name == "Movies":
            r = np.vstack([r_rows("Sherlock", "movies"), r_rows("FilmFest", "movies")])
        else:
            r = R_ALL[name][0]
        m = source_model(name)
        w = np.zeros(r.shape[1])
        w[m["sig"]] = m["signs"]
        np.save(os.path.join(CPM_DIR, f"{name}_r_matrix_{ATLAS}.npy"), r)
        np.save(os.path.join(CPM_DIR, f"{name}_binarized_weights_{ATLAS}_{ALPHA:g}.npy"), w)
        print(f"{name:12s} r_matrix {r.shape}   weights: {int((w > 0).sum())} positive, {int((w < 0).sum())} negative")
    print("Saved to", CPM_DIR)